 Cupcakes Shop

In [31]:
import simpy
import random
import math

In [32]:
class Cupcakes:
    
    def __init__(self,env,minStock,maxStock):
        self.env=env
        
        # Inventory Variables
        self.currentStock=maxStock
        self.s=minStock
        self.S=maxStock

        self.incoming_orders = []        # list of {quantity, delivery_day}
        
        # Tracking Variables (for math later)
        self.total_holding_cost = 0
        self.total_order_cost = 0
        self.stockout_days = 0
        
        # check for order variable
        self.order_in_transit = False
        self.orderArrival=0

        # for average inventory calculation
        self.total_stock_sum = 0   
    
    def runDailyOperation(self):

        current_day = 0
        while True:
            # Check arrivals
            arrived = [o for o in self.incoming_orders if o['delivery_day'] <= current_day]
            for order in arrived:
                self.currentStock += order['quantity']
                self.incoming_orders.remove(order)
            if not self.incoming_orders:
                self.order_in_transit = False

            # Generate demand
            todayDemand=int(max(0,random.gauss(50,10)))
            
            #Process sales
            if todayDemand<=self.currentStock:
                self.currentStock-=todayDemand
            else:
                self.currentStock=0
                self.stockout_days+=1

            # Track inventory for avg calculation
            self.total_stock_sum += self.currentStock

            # Order decision
            if self.currentStock<=self.s and self.order_in_transit==False:
                self.total_order_cost+=50
                orderAmount=self.S-self.currentStock
                self.env.process(self.order(orderAmount, current_day))

            current_day+=1
            yield self.env.timeout(1) 
                
    def order(self,amount, order_day):
        self.order_in_transit=True
        days=random.randint(2,5)
        delivery_day=order_day+days
        self.incoming_orders.append({'quantity': amount, 'delivery_day': delivery_day})
        yield self.env.timeout(days)
        self.order_in_transit=False
    

In [33]:
def Simulate(s,S,daysToRun=100):
    env=simpy.Environment()
    
    myCupcakeStore= Cupcakes(env,s,S)
    
    env.process(myCupcakeStore.runDailyOperation())
    
    env.run(until=daysToRun)
    
    serviceLevel=(int(daysToRun)-int(myCupcakeStore.stockout_days))/int(daysToRun)
    avg_inventory = myCupcakeStore.total_stock_sum / daysToRun
    holding_cost = 5 * avg_inventory
    totalCost=int(holding_cost)+int(myCupcakeStore.total_order_cost)+(int(myCupcakeStore.stockout_days)*100)
    
    print(f"Service Level  : {serviceLevel*100:.1f}%")
    print(f"Holding Cost   : ${holding_cost:.0f}")
    print(f"Ordering Cost  : ${myCupcakeStore.total_order_cost}")
    print(f"Stockout Cost  : ${myCupcakeStore.stockout_days * 100}")
    print(f"Total Cost     : ${totalCost:.0f}")

In [34]:
Simulate(40,100)

Service Level  : 30.0%
Holding Cost   : $46
Ordering Cost  : $1300
Stockout Cost  : $7000
Total Cost     : $8346


In [35]:
# Create a list of the test values
s_values = [30, 35, 40, 45]
max_stock_S = 100

print("=== STARTING OPTIMIZATION ===")

for s_val in s_values:
    print(f"\n--- Testing Reorder Point (s) = {s_val} ---")
    
    Simulate(s_val, max_stock_S)

print("\n=== OPTIMIZATION COMPLETE ===")

=== STARTING OPTIMIZATION ===

--- Testing Reorder Point (s) = 30 ---
Service Level  : 36.0%
Holding Cost   : $57
Ordering Cost  : $1300
Stockout Cost  : $6400
Total Cost     : $7756

--- Testing Reorder Point (s) = 35 ---
Service Level  : 36.0%
Holding Cost   : $56
Ordering Cost  : $1300
Stockout Cost  : $6400
Total Cost     : $7756

--- Testing Reorder Point (s) = 40 ---
Service Level  : 34.0%
Holding Cost   : $48
Ordering Cost  : $1300
Stockout Cost  : $6600
Total Cost     : $7948

--- Testing Reorder Point (s) = 45 ---
Service Level  : 29.0%
Holding Cost   : $38
Ordering Cost  : $1250
Stockout Cost  : $7100
Total Cost     : $8388

=== OPTIMIZATION COMPLETE ===
